# UIT-ViQuAD2.0 — Qwen2.5-1.5B: train 3 adapters công bằng

Train **tuần tự** (không train 1 lần sinh 3) với **cùng data / prompt / seed / epochs**:

1. **LoRA** → `qwen2.5-1.5b-instruct-lora-viquad2`
2. **TinyLoRA (strict)** → `qwen2.5-1.5b-instruct-tinylora-viquad2`  
   _(không fallback; nếu PEFT không tương thích sẽ báo lỗi để bạn sửa version)_
3. **DoRA** (Decomposed LoRA / "DeLoRA"-style) → `qwen2.5-1.5b-instruct-dora-viquad2`

Sau đó **1 block eval** so sánh EM / F1 / NoAns trên cùng test set.

### Cách chạy
1. Pip cells → **Restart Kernel**
2. Warnings → Config → Data → Profiling → Dataset format
3. Chạy **Train All Variants** (3 lần train tuần tự, ~3× thời gian 1 LoRA)
4. Chạy **Compare Evaluation**

In [ ]:
!pip uninstall torch torchvision torchaudio xformers transformers trl unsloth unsloth_zoo -y

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir

In [ ]:
# TinyLoRA cần PEFT >= 0.19 (API: r, u, tinylora_dropout — không có lora_alpha)
!pip install "peft>=0.19.0" transformers trl accelerate bitsandbytes xformers datasets --no-cache-dir
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-cache-dir
!pip install unsloth_zoo scikit-learn --no-cache-dir

In [ ]:
import importlib
import inspect

for pkg in ["torch", "transformers", "datasets", "unsloth", "peft"]:
    importlib.import_module(pkg)
    print(f"OK  {pkg}")

import peft
from peft import TinyLoraConfig

print(f"PEFT version: {peft.__version__}")
sig = inspect.signature(TinyLoraConfig.__init__).parameters
print("TinyLoraConfig params:", ", ".join(k for k in sig if k != "self"))
if "u" in sig:
    print("TinyLoRA API: modern (u, tinylora_dropout)")
elif "num_projections" in sig:
    print("TinyLoRA API: legacy (num_projections)")
else:
    print("WARNING: TinyLoraConfig signature không nhận diện được — cần peft>=0.19")

In [ ]:
import warnings, logging, os, sys
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore"
for _n in ("transformers", "datasets", "torch", "unsloth", "peft", "accelerate", "huggingface_hub"):
    logging.getLogger(_n).setLevel(logging.ERROR)
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass
print("Warnings suppressed.")

In [ ]:
import json
import math
import gc
import re
import string
import unicodedata
from collections import Counter
from pathlib import Path

import torch
from datasets import load_dataset, Dataset
from tqdm import tqdm
from transformers import AutoTokenizer

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# ========== SHARED CONFIG (giữ cố định cho 3 method) ==========
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_NAME = "taidng/UIT-ViQuAD2.0"
NO_ANSWER_SENTINEL = "Không có đáp án trong đoạn văn"

SYSTEM_PROMPT = (
    "Bạn là hệ thống trích xuất câu trả lời từ văn bản tiếng Việt. "
    "QUY TẮC BẮT BUỘC:\n"
    "1) Chỉ trả về ĐÚNG cụm từ xuất hiện trong đoạn văn, không thêm từ nào khác.\n"
    "2) Không viết câu hoàn chỉnh, không giải thích, không thêm 'Đáp án:'.\n"
    f"3) Nếu không có đáp án trong đoạn văn, chỉ trả về: {NO_ANSWER_SENTINEL}"
)
USER_PROMPT_TEMPLATE = (
    "Trích xuất câu trả lời từ đoạn văn. Chỉ trả về cụm từ trong đoạn văn.\n\n"
    "Đoạn văn:\n{context}\n\n"
    "Câu hỏi: {question}\n\n"
    f"Câu trả lời (span-only hoặc '{NO_ANSWER_SENTINEL}'):"
)

DATASET_ROOT = Path("../../")
DEV_JSON_PATH = DATASET_ROOT / "dev_viquad2.json"
TEST_JSON_PATH = DATASET_ROOT / "test_viquad2.json"
PROFILING_CONFIG_PATH = "profiling_config.json"
COMPARE_EVAL_PATH = "eval_compare_adapters_viquad2.json"

RUN_TRAINING = True
RUN_EVALUATION = True
FORCE_HF_DATASET = True
FORCE_REEXPORT_JSON = True

LOAD_IN_4BIT = False
LOAD_IN_8BIT = False
MAX_SEQ_CAP = 4096
MIN_SEQ_LENGTH = 512
MAX_NEW_TOKENS = 32
USE_SPAN_POSTPROCESS = True

# Eval: None = full test (~1h); 100 = smoke test nhanh
EVAL_MAX_SAMPLES = 100
EVAL_LOG_EVERY = 10
USE_UNSLOTH_FOR_INFERENCE = False  # True hay treo sau Adapter OK trên một số setup

# Cùng hyperparams train cho 3 method (fairness)
TRAIN_COMMON = dict(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    num_train_epochs=3,
    learning_rate=2e-4,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
)

TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# 3 variants — train tuần tự
ADAPTER_VARIANTS = [
    {
        "name": "lora",
        "save_path": "qwen2.5-1.5b-instruct-lora-viquad2",
        "output_dir": "outputs_viquad2_lora",
    },
    {
        "name": "tinylora",
        "save_path": "qwen2.5-1.5b-instruct-tinylora-viquad2",
        "output_dir": "outputs_viquad2_tinylora",
    },
    {
        "name": "dora",  # DoRA = Decomposed LoRA (thường gọi DeLoRA)
        "save_path": "qwen2.5-1.5b-instruct-dora-viquad2",
        "output_dir": "outputs_viquad2_dora",
    },
]

# Bật/tắt từng method nếu muốn train 1 cái trước
TRAIN_METHODS = ["lora", "tinylora", "dora"]  # hoặc ["lora"] để test nhanh
EVAL_METHODS = ["lora", "tinylora", "dora"]

TQDM_BAR = "{desc}: {percentage:3.0f}%|{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"


def build_messages(context, question, answer=None, for_inference=False):
    user_content = USER_PROMPT_TEMPLATE.format(context=context, question=question)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    if answer is not None and not for_inference:
        messages.append({"role": "assistant", "content": answer})
    return messages


def sample_to_train_text(sample, tok):
    messages = build_messages(sample["context"], sample["question"], sample["answer"])
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def load_tokenizer(model_path=BASE_MODEL_NAME):
    tok = AutoTokenizer.from_pretrained(model_path)
    if tok.chat_template is None:
        raise RuntimeError(f"Tokenizer {model_path} thiếu chat_template.")
    return tok


def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

In [ ]:
def row_to_sample(row):
    is_impossible = bool(row["is_impossible"])
    texts = (row.get("answers") or {}).get("text") or []
    if is_impossible or len(texts) == 0 or not str(texts[0] if texts else "").strip():
        return {
            "id": row.get("id", ""), "title": row.get("title", ""),
            "context": row["context"], "question": row["question"],
            "answer": NO_ANSWER_SENTINEL, "is_impossible": True,
        }
    return {
        "id": row.get("id", ""), "title": row.get("title", ""),
        "context": row["context"], "question": row["question"],
        "answer": texts[0], "is_impossible": False,
    }


def hf_split_to_samples(split_dataset):
    return [row_to_sample(row) for row in split_dataset]


def save_samples_json(samples, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(samples)} → {path.resolve()}")


def split_stats(name, samples):
    n = len(samples)
    n_no = sum(1 for s in samples if s["is_impossible"])
    print(f"{name}: {n} | NoAns: {n_no} ({100*n_no/max(n,1):.1f}%)")


try:
    raw_ds = load_dataset(DATASET_NAME)
    train_samples = hf_split_to_samples(raw_ds["train"])
    dev_samples = hf_split_to_samples(raw_ds["validation"])
    test_samples = hf_split_to_samples(raw_ds["test"])
except Exception as e:
    if FORCE_HF_DATASET:
        raise RuntimeError(f"Không tải được HF: {e}")
    with open(TEST_JSON_PATH, encoding="utf-8") as f:
        test_samples = json.load(f)
    with open(DEV_JSON_PATH, encoding="utf-8") as f:
        dev_samples = json.load(f)
    train_samples = dev_samples

split_stats("Train", train_samples)
split_stats("Val", dev_samples)
split_stats("Test", test_samples)
if FORCE_REEXPORT_JSON:
    save_samples_json(dev_samples, DEV_JSON_PATH)
    save_samples_json(test_samples, TEST_JSON_PATH)

In [ ]:
if "train_samples" not in globals():
    raise NameError("Chạy cell tải dataset trước.")


def compute_max_seq_length(samples, tok, cap=MAX_SEQ_CAP, min_len=MIN_SEQ_LENGTH):
    lengths = []
    total = len(samples)
    pbar = tqdm(samples, total=total, desc="Token profiling", unit="sample", bar_format=TQDM_BAR)
    for i, s in enumerate(pbar, 1):
        lengths.append(len(tok.encode(sample_to_train_text(s, tok))))
        pbar.set_postfix(done=f"{i}/{total}")
    lengths.sort()
    n = len(lengths)
    stats = {
        "min": lengths[0], "p50": lengths[n // 2],
        "p95": lengths[int(n * 0.95)], "p99": lengths[int(n * 0.99)], "max": lengths[-1],
    }
    chosen = max(((min(math.ceil(stats["p99"] * 1.05), cap) + 255) // 256) * 256, min_len)
    truncated = sum(1 for L in lengths if L > chosen)
    stats.update({"chosen_max_seq_length": chosen, "truncated_samples": truncated,
                  "truncated_pct": round(100 * truncated / n, 3)})
    return chosen, stats


tokenizer_prof = load_tokenizer()
if Path(PROFILING_CONFIG_PATH).exists() and not RUN_TRAINING:
    prof_cfg = json.load(open(PROFILING_CONFIG_PATH, encoding="utf-8"))
    max_seq_length = prof_cfg["max_seq_length"]
    length_stats = prof_cfg["token_length_stats"]
else:
    max_seq_length, length_stats = compute_max_seq_length(train_samples, tokenizer_prof)
    json.dump({"max_seq_length": max_seq_length, "token_length_stats": length_stats},
              open(PROFILING_CONFIG_PATH, "w", encoding="utf-8"), indent=2)
print(f"max_seq_length = {max_seq_length}")
for k, v in length_stats.items():
    print(f"  {k}: {v}")

In [ ]:
# Format dataset 1 lần — dùng chung cho cả 3 method
tokenizer_fmt = load_tokenizer()

def formatting_prompts_func(examples):
    texts = []
    for ctx, q, ans in zip(examples["context"], examples["question"], examples["answer"]):
        msgs = build_messages(ctx, q, ans)
        texts.append(tokenizer_fmt.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

train_hf = Dataset.from_list(train_samples)
dataset = train_hf.map(formatting_prompts_func, batched=True, remove_columns=train_hf.column_names)
print(f"Shared train dataset: {len(dataset)} samples")
print(dataset[0]["text"][:400])

## Train 3 adapters tuần tự (công bằng)

Mỗi method: load base → gắn adapter → `SFTTrainer.train()` → `save_pretrained` → giải phóng GPU → sang method tiếp.

Chỉnh `TRAIN_METHODS` nếu chỉ muốn train 1–2 cái trước.

In [ ]:
def apply_adapter(model, method_name):
    """Gắn đúng PEFT method lên base model đã load bằng Unsloth."""
    from unsloth import FastLanguageModel

    def _resolve_causallm(m):
        """TinyLoRA/PEFT cần ForCausalLM — KHÔNG unwrap .model (Qwen2Model)."""
        if hasattr(m, "prepare_inputs_for_generation"):
            return m
        if hasattr(m, "get_base_model"):
            base = m.get_base_model()
            if hasattr(base, "prepare_inputs_for_generation"):
                return base
        raise RuntimeError(
            "Không tìm thấy ForCausalLM. Đừng dùng model.model — đó là Qwen2Model "
            "(thiếu prepare_inputs_for_generation)."
        )

    if method_name == "lora":
        model = FastLanguageModel.get_peft_model(
            model,
            r=16, lora_alpha=32, target_modules=TARGET_MODULES,
            lora_dropout=0.05, bias="none",
            use_gradient_checkpointing="unsloth", random_state=3407,
            use_dora=False,
        )
        print("Applied: LoRA (r=16, alpha=32)")
        return model

    if method_name == "dora":
        model = FastLanguageModel.get_peft_model(
            model,
            r=16, lora_alpha=32, target_modules=TARGET_MODULES,
            lora_dropout=0.05, bias="none",
            use_gradient_checkpointing="unsloth", random_state=3407,
            use_dora=True,  # DoRA / Decomposed LoRA
        )
        print("Applied: DoRA (r=16, alpha=32, use_dora=True)")
        return model

    if method_name == "tinylora":
        import inspect
        import peft
        from peft import TinyLoraConfig, get_peft_model as peft_get_model

        # TinyLoRA (PEFT >= 0.19): r, u, tinylora_dropout — KHÔNG dùng lora_alpha/lora_dropout
        sig = inspect.signature(TinyLoraConfig.__init__).parameters
        desired = {
            "r": 2,                      # paper khuyến nghị r=2
            "u": 64,                     # modern API: trainable vector dim
            "num_projections": 64,       # legacy API alias của u
            "target_modules": TARGET_MODULES,
            "tinylora_dropout": 0.0,
            "lora_dropout": 0.0,         # chỉ dùng nếu PEFT cũ còn param này
            "bias": "none",
            "task_type": "CAUSAL_LM",
            "weight_tying": 0.0,
            "projection_seed": 3407,
            "init_weights": True,
        }
        tiny_kwargs = {k: v for k, v in desired.items() if k in sig}
        if "target_modules" not in tiny_kwargs:
            tiny_kwargs["target_modules"] = TARGET_MODULES

        try:
            tinylora_config = TinyLoraConfig(**tiny_kwargs)
        except Exception as e:
            raise RuntimeError(
                f"TinyLoRA không khả dụng với PEFT {peft.__version__}: {e}. "
                "Chạy lại pip cell (peft>=0.19) rồi Restart Kernel."
            ) from e

        causallm = _resolve_causallm(model)
        model = peft_get_model(causallm, tinylora_config)
        model.config.use_cache = False
        if hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable()
        model.print_trainable_parameters()
        u_val = tiny_kwargs.get("u", tiny_kwargs.get("num_projections", "?"))
        print(f"Applied: TinyLoRA (PEFT {peft.__version__}, r={tiny_kwargs.get('r')}, u={u_val})")
        return model

    raise ValueError(f"Unknown method: {method_name}")


def train_one_variant(variant, max_seq_length, dataset):
    from unsloth import FastLanguageModel, is_bfloat16_supported
    from trl import SFTTrainer
    from transformers import TrainingArguments
    import sys

    name = variant["name"]
    print("\n" + "=" * 60)
    print(f" TRAIN VARIANT: {name.upper()}")
    print("=" * 60)

    clear_gpu()
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_NAME,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=LOAD_IN_4BIT,
        load_in_8bit=LOAD_IN_8BIT,
    )
    model = apply_adapter(model, name)

    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        dataset_num_proc=1,
        packing=False,
        args=TrainingArguments(
            **TRAIN_COMMON,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            output_dir=variant["output_dir"],
            # Giữ progress bar 1 dòng, giảm log dài
            disable_tqdm=False,
            logging_strategy="steps",
            logging_steps=200,
            logging_first_step=False,
            log_level="error",
            log_level_replica="error",
            # Lưu checkpoint mỗi 200 steps để ngắt giữa chừng vẫn resume được
            save_strategy="steps",
            save_steps=200,
            save_total_limit=3,
            report_to="none",
        ),
    )
    cfg_cls, tr_cls = type(trainer.args), type(trainer)
    sys.modules["trl.trainer.sft_config"] = sys.modules[cfg_cls.__module__]
    sys.modules["trl.trainer.sft_trainer"] = sys.modules[tr_cls.__module__]
    sys.modules[cfg_cls.__module__].SFTConfig = cfg_cls
    sys.modules[tr_cls.__module__].SFTTrainer = tr_cls

    effective_batch = trainer.args.per_device_train_batch_size * trainer.args.gradient_accumulation_steps
    steps_per_epoch = math.ceil(len(dataset) / max(effective_batch, 1))
    total_steps = int(steps_per_epoch * trainer.args.num_train_epochs)
    print(f"Planned: epochs={trainer.args.num_train_epochs}, steps/epoch≈{steps_per_epoch}, total_steps≈{total_steps}")

    trainer.train()
    Path(variant["save_path"]).mkdir(parents=True, exist_ok=True)
    model.save_pretrained(variant["save_path"])
    tokenizer.save_pretrained(variant["save_path"])
    print(f"Saved adapter → {variant['save_path']}")

    # Giải phóng trước khi train method tiếp theo
    del trainer, model, tokenizer
    clear_gpu()
    return variant["save_path"]

In [ ]:
if RUN_TRAINING:
    variant_map = {v["name"]: v for v in ADAPTER_VARIANTS}
    trained_paths = {}
    for method in TRAIN_METHODS:
        if method not in variant_map:
            raise ValueError(f"Unknown TRAIN_METHODS item: {method}")
        path = train_one_variant(variant_map[method], max_seq_length, dataset)
        trained_paths[method] = path
    print("\n✅ Train xong tất cả methods:")
    for k, v in trained_paths.items():
        print(f"  {k}: {v}")
else:
    print("RUN_TRAINING=False — bỏ qua train.")

## Compare Evaluation — LoRA vs TinyLoRA vs DoRA

Cùng test set, cùng post-process, in bảng EM/F1/NoAns.

In [ ]:
PREFIX_RE = re.compile(
    r"^(đáp án|answer|câu trả lời|theo đoạn văn|trong đoạn văn)\s*[:\-]?\s*",
    re.IGNORECASE,
)

def normalize_text(text):
    text = unicodedata.normalize("NFC", text or "")
    return " ".join(text.lower().translate(str.maketrans("", "", string.punctuation)).split())

def compute_em(pred, truth):
    return int(normalize_text(pred) == normalize_text(truth))

def compute_f1(pred, truth):
    pt, tt = normalize_text(pred).split(), normalize_text(truth).split()
    if not pt and not tt: return 1.0
    if not pt or not tt: return 0.0
    common = Counter(pt) & Counter(tt)
    n = sum(common.values())
    if n == 0: return 0.0
    p, r = n / len(pt), n / len(tt)
    return 2 * p * r / (p + r)

def is_no_answer(pred):
    return normalize_text(pred) == normalize_text(NO_ANSWER_SENTINEL)

def clean_prediction(raw):
    pred = raw.strip().split("\n")[0].strip().strip('"\'')
    return PREFIX_RE.sub("", pred).strip()


def align_to_context(pred, context):
    if not pred or is_no_answer(pred):
        return pred
    idx = context.lower().find(pred.lower())
    if idx >= 0:
        return context[idx:idx + len(pred)]
    words, pred_words = context.split(), normalize_text(pred).split()
    if not pred_words:
        return pred
    n = len(pred_words)
    best_span, best_f1 = pred, 0.0
    for i in range(len(words)):
        for j in range(i + 1, min(i + n + 4, len(words)) + 1):
            span = " ".join(words[i:j])
            f1 = compute_f1(span, pred)
            if f1 > best_f1:
                best_f1, best_span = f1, span
    return best_span.strip() if best_f1 >= 0.5 else pred


def _read_adapter_base_model(adapter_path):
    cfg_path = Path(adapter_path) / "adapter_config.json"
    if not cfg_path.exists():
        raise FileNotFoundError(f"Thiếu adapter_config.json trong {adapter_path}")
    cfg = json.load(open(cfg_path, encoding="utf-8"))
    return cfg.get("base_model_name_or_path", BASE_MODEL_NAME)


def _validate_adapter_files(adapter_path):
    adapter_path = Path(adapter_path)
    st_path = adapter_path / "adapter_model.safetensors"
    if not st_path.exists():
        raise FileNotFoundError(f"Thiếu {st_path}")

    head = open(st_path, "rb").read(200)
    if head.startswith(b"version https://git-lfs"):
        raise RuntimeError(
            f"{st_path.name} là Git LFS pointer — chưa pull weights thật. "
            "Chạy: git lfs pull"
        )

    size_mb = st_path.stat().st_size / (1024 * 1024)
    if size_mb < 0.1:
        raise RuntimeError(f"{st_path.name} quá nhỏ ({size_mb:.2f} MB) — file corrupt.")

    # Thử mở safetensors trước (bắt lỗi sớm)
    from safetensors import safe_open
    try:
        with safe_open(str(st_path), framework="pt", device="cpu") as f:
            _ = f.keys()
    except Exception as e:
        raise RuntimeError(f"{st_path.name} không đọc được: {e}") from e

    print(f"Adapter file OK: {st_path.name} ({size_mb:.1f} MB)")


def load_eval_model(adapter_path, max_seq_length):
    """Load Unsloth base (GPU) + gắn LoRA — không roundtrip CPU→GPU (tránh treo 1h)."""
    import torch
    from unsloth import FastLanguageModel
    from peft import PeftModel

    _validate_adapter_files(adapter_path)
    base_model = _read_adapter_base_model(adapter_path)
    print(f"Eval load: base={base_model} | adapter={adapter_path}", flush=True)

    load_dtype = torch.float16 if not (LOAD_IN_4BIT or LOAD_IN_8BIT) else None

    print("[Eval] Loading base model on GPU...", flush=True)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=base_model,
        max_seq_length=max_seq_length,
        dtype=load_dtype,
        load_in_4bit=LOAD_IN_4BIT,
        load_in_8bit=LOAD_IN_8BIT,
    )
    print("[Eval] Base loaded.", flush=True)

    # Chỉ load adapter weights trên CPU, giữ base trên GPU (không model.cpu())
    print("[Eval] Attaching LoRA adapter...", flush=True)
    model = PeftModel.from_pretrained(
        model,
        adapter_path,
        is_trainable=False,
        torch_device="cpu",
    )
    if not hasattr(model, "peft_config") or not model.peft_config:
        raise RuntimeError("LoRA adapter KHÔNG được gắn.")
    print(f"Adapter OK: {list(model.peft_config.keys())}", flush=True)

    if USE_UNSLOTH_FOR_INFERENCE:
        print("[Eval] Unsloth for_inference...", flush=True)
        FastLanguageModel.for_inference(model)
    else:
        print("[Eval] Skip for_inference (dùng model.eval() thường).", flush=True)

    model.config.use_cache = True
    model.eval()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("[Eval] Model ready — bắt đầu generate.", flush=True)
    return model, tokenizer

In [ ]:
def eval_one_adapter(method_name, adapter_path, test_samples, max_seq_length):
    import time
    import sys
    from tqdm.auto import tqdm

    print(f"\n--- Evaluating: {method_name} | {adapter_path} ---", flush=True)
    if not Path(adapter_path).exists():
        print(f"⚠ SKIP {method_name}: adapter path không tồn tại.", flush=True)
        return None

    clear_gpu()
    model_eval, tokenizer_eval = load_eval_model(adapter_path, max_seq_length)
    model_eval.generation_config.max_length = None
    model_eval.generation_config.max_new_tokens = MAX_NEW_TOKENS

    samples = test_samples[:EVAL_MAX_SAMPLES] if EVAL_MAX_SAMPLES else test_samples
    total = len(samples)
    print(f"[Eval] Starting {total} samples (max_seq={max_seq_length})...", flush=True)

    has_em, has_f1, no_ok, preds = [], [], [], []
    t0 = time.time()
    pbar = tqdm(samples, total=total, desc=f"Eval-{method_name}", bar_format=TQDM_BAR, file=sys.stdout, mininterval=1.0)

    for i, s in enumerate(pbar, 1):
        msgs = build_messages(s["context"], s["question"], for_inference=True)
        prompt = tokenizer_eval.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer_eval(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=max_seq_length,
        ).to(model_eval.device)

        if i == 1:
            print(f"[Eval] Running first sample (warmup)...", flush=True)
        t_gen = time.time()

        with torch.no_grad():
            out = model_eval.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs.get("attention_mask"),
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer_eval.pad_token_id,
                eos_token_id=tokenizer_eval.eos_token_id,
            )

        if i == 1:
            print(f"[Eval] First sample done in {time.time()-t_gen:.1f}s", flush=True)

        raw = tokenizer_eval.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        pred_raw = clean_prediction(raw)
        pred = align_to_context(pred_raw, s["context"]) if USE_SPAN_POSTPROCESS else pred_raw

        if s["is_impossible"]:
            ok = int(is_no_answer(pred))
            no_ok.append(ok)
            em, f1 = ok, float(ok)
        else:
            em = compute_em(pred, s["answer"])
            f1 = compute_f1(pred, s["answer"])
            has_em.append(em)
            has_f1.append(f1)

        preds.append({
            "id": s["id"],
            "question": s["question"],
            "is_impossible": s["is_impossible"],
            "ground_truth": s["answer"],
            "prediction_raw": pred_raw,
            "prediction": pred,
            "em": em,
            "f1": f1,
        })

        if has_em:
            pbar.set_postfix(em=f"{100*sum(has_em)/len(has_em):.1f}%")

        if i == 1 or i % EVAL_LOG_EVERY == 0 or i == total:
            elapsed = time.time() - t0
            rate = i / max(elapsed, 0.001)
            eta = (total - i) / max(rate, 0.001)
            print(f"[Eval] {i}/{total} | {elapsed/60:.1f}m elapsed | ETA {eta/60:.1f}m | {rate:.2f} sample/s", flush=True)

    pbar.close()
    print(f"[Eval] Done {total} samples in {(time.time()-t0)/60:.1f} min", flush=True)

    n_has, n_no = len(has_em), len(no_ok)
    metrics = {
        "method": method_name,
        "adapter": adapter_path,
        "hasans_em": round(100 * sum(has_em) / max(n_has, 1), 4),
        "hasans_f1": round(100 * sum(has_f1) / max(n_has, 1), 4),
        "noans_accuracy": round(100 * sum(no_ok) / n_no, 4) if n_no else None,
        "n_hasans": n_has,
        "n_noans": n_no,
        "total": total,
    }

    del model_eval, tokenizer_eval
    clear_gpu()
    return {"metrics": metrics, "predictions": preds}

In [ ]:
if RUN_EVALUATION:
    variant_map = {v["name"]: v for v in ADAPTER_VARIANTS}
    all_results = {}

    for method in EVAL_METHODS:
        path = variant_map[method]["save_path"]
        result = eval_one_adapter(method, path, test_samples, max_seq_length)
        if result is not None:
            all_results[method] = result

    # ===== Comparison table =====
    line = "=" * 72
    print("\n" + line)
    print("  SO SÁNH ADAPTERS — UIT-ViQuAD2.0 | Qwen2.5-1.5B-Instruct")
    print(line)
    print(f"{'Method':<12} {'HasAns EM':>12} {'HasAns F1':>12} {'NoAns Acc':>12} {'n_has/n_no':>14}")
    print("-" * 72)
    for method, result in all_results.items():
        m = result["metrics"]
        noans = f"{m['noans_accuracy']:.2f}%" if m["noans_accuracy"] is not None else "N/A"
        print(f"{method:<12} {m['hasans_em']:>11.2f}% {m['hasans_f1']:>11.2f}% {noans:>12} "
              f"{m['n_hasans']}/{m['n_noans']:>6}")
    print(line)

    # Pretty report từng method (F1 buckets)
    for method, result in all_results.items():
        m = result["metrics"]
        has_f1 = [p["f1"] for p in result["predictions"] if not p["is_impossible"]]
        buckets = {"0-0.2": 0, "0.2-0.5": 0, "0.5-0.8": 0, "0.8-1.0": 0, "1.0": 0}
        for f1 in has_f1:
            if f1 == 1.0: buckets["1.0"] += 1
            elif f1 >= 0.8: buckets["0.8-1.0"] += 1
            elif f1 >= 0.5: buckets["0.5-0.8"] += 1
            elif f1 >= 0.2: buckets["0.2-0.5"] += 1
            else: buckets["0-0.2"] += 1

        print(f"\n--- {method.upper()} detail ---")
        print(f"Exact Match (HasAns): {m['hasans_em']:.2f}%")
        print(f"F1-score (HasAns)   : {m['hasans_f1']:.2f}%")
        if m["n_hasans"]:
            print("Phân phối F1-score (HasAns):")
            for b, c in buckets.items():
                print(f"  F1 = {b}: {c} câu ({100*c/m['n_hasans']:.1f}%)")

    # Save compare JSON (predictions rút gọn để file nhẹ hơn)
    save_payload = {
        "dataset": DATASET_NAME,
        "base_model": BASE_MODEL_NAME,
        "max_seq_length": max_seq_length,
        "train_common": TRAIN_COMMON,
        "summary": {k: v["metrics"] for k, v in all_results.items()},
        "predictions": {k: v["predictions"] for k, v in all_results.items()},
    }
    with open(COMPARE_EVAL_PATH, "w", encoding="utf-8") as f:
        json.dump(save_payload, f, ensure_ascii=False, indent=2)
    print(f"\nSaved comparison → {COMPARE_EVAL_PATH}")
else:
    print("RUN_EVALUATION=False — bỏ qua eval.")